# Metrics and Evaluation - California Housing Dataset

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

x, y = fetch_california_housing(return_X_y=True,as_frame=True)

x.head()

In [ ]:


x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [ ]:
cols = set(x.columns) - {"Longitude", "Latitude"}
Q1 = x_train.quantile(0.25)
Q3 = x_train.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
x_train_tensorial = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensorial = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

x_val_tensorial = torch.tensor(x_val.values, dtype=torch.float32)
y_val_tensorial = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

x_test_tensorial = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensorial = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [ ]:
mean = torch.mean(x_train_tensorial, dim=0)
std = torch.std(x_train_tensorial, dim=0)
std[std == 0] = 2.220446049250313e-16 # no me va a pasar DOS veces, por eso el epsilon

In [ ]:
class modelo_hotel_california(nn.Module):
    def __init__(self, entrada, n_capas):
        super().__init__()
        
        self.mean = mean
        self.std = std
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound

        self.red_californiana = nn.Sequential(
            nn.Linear(entrada, n_capas),
            nn.ReLU(),
            nn.Linear(n_capas, 1) # NO ERA SOFTMAX??
        )

    def forward(self, x):
        for i in range(x.shape[1]):
            x[:,i] = torch.clamp(x, min=self.lower_bound, max=upper_bound)
            x[:,i] = (x[:,i] - self.mean) / self.mean
        return self.red_californiana(x)
   

In [ ]:
def entrenar(x_train_tensorial, y_train_tensorial, x_val_tensorial, y_val_tensorial, n_capas, lr):
    modelo_californiano = modelo_hotel_california(x_train_tensorial.shape[1], n_capas)

    sandler_californiano = optim.Adam(modelo_californiano.parameters(), lr = lr)
    loss_fn = nn.MSELoss()

    for _ in range(50):

        sandler_californiano.zero_grad()
        # LO CAMBIA TODO
        loss = loss_fn(modelo_californiano(x_train_tensorial), y_train_tensorial)
        loss.backwards()
        
        sandler_californiano.step()
        
        # ya me lo aprendi

    with torch.no_grad():
        val_loss = loss_fn(modelo_californiano(x_val_tensorial), y_val_tensorial).item()
        
    return modelo_californiano, val_loss
   

In [ ]:
resultados = []

for configuracion_californiana in [(64),(32),(128),(320),()]: #esto esta mas limpio que ponerle 3000 celdas con MD apoco no profe #yodo: especular pero con los valores
    modelo_californiano, val_loss = entrenar(
        x_train_tensorial, y_train_tensorial,
        x_val_tensorial, y_val_tensorial,
        configuracion_californiana[0],
        configuracion_californiana[1]
    )

    resultados.append((modelo_californiano, val_loss, configuracion_californiana))
    print(configuracion_californiana, val_loss)

In [ ]:
california_si_no_especulara, _, configuracion_californiana = min(resultados, key=lambda x: x[1])

print("Mejor modelo", configuracion_californiana)

In [ ]:
with torch.no_grad():
    test_loss = nn.MSELoss()(california_si_no_especulara(x_test_tensorial), y_test_tensorial).item()

print("Evaliacion", test_loss)